# Blender GPU Benchmark — ChatGPT Work

Prueba reproducible para comparar nuestros renders de Blender en Google Colab frente a GitHub Actions CPU.

**Antes de ejecutar:** en Colab abre `Entorno de ejecución > Cambiar tipo de entorno de ejecución` y selecciona **GPU** si está disponible.

El notebook usa por defecto **Blender Neon Core** y renderiza el proyecto real del repositorio. Al terminar copia `out/` a la carpeta correspondiente de Google Drive.

In [ ]:
# 1) Comprobar hardware asignado por Colab
import os, subprocess, textwrap, json, pathlib, time
print(subprocess.run(['bash','-lc','nvidia-smi || true'], capture_output=True, text=True).stdout)
print('CPU:', subprocess.run(['bash','-lc','lscpu | grep "Model name" | head -1'], capture_output=True, text=True).stdout.strip())

In [ ]:
# 2) Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Descargar workspace
REPO='https://github.com/izquierdodavid528-code/Chat-gpt-work-.git'
!rm -rf /content/Chat-gpt-work-
!git clone --depth 1 "$REPO" /content/Chat-gpt-work-
%cd /content/Chat-gpt-work-

In [ ]:
# 4) Instalar Blender oficial 4.2 LTS
BLENDER_VERSION='4.2.5'
BLENDER_DIR=f'/content/blender-{BLENDER_VERSION}-linux-x64'
BLENDER_BIN=f'{BLENDER_DIR}/blender'
if not os.path.exists(BLENDER_BIN):
    url=f'https://download.blender.org/release/Blender4.2/blender-{BLENDER_VERSION}-linux-x64.tar.xz'
    !wget -q --show-progress "$url" -O /content/blender.tar.xz
    !tar -xf /content/blender.tar.xz -C /content
print('Blender:', BLENDER_BIN)
!$BLENDER_BIN --version | head -n 3

In [ ]:
# 5) Preflight seguro para modo headless
# En Blender -b, gpu.platform.renderer_get() puede fallar porque las funciones de dibujo
# no están disponibles. Eso NO significa que Colab no tenga GPU.
import subprocess
print(subprocess.run(['bash','-lc','nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader'], capture_output=True, text=True).stdout.strip())
print('Nota: el uso real de la GPU se validará durante el render de EEVEE en el paso 7.')


In [ ]:
# 6) Elegir proyecto
PROJECT_SLUG='blender-neon-core-demo'  # cambia a 'haaland-goal-cinematic-test' para la otra prueba
PROJECT_DIR=f'/content/Chat-gpt-work-/projects/{PROJECT_SLUG}'
cfg=json.load(open(f'{PROJECT_DIR}/project.config.json'))
print(json.dumps(cfg, indent=2, ensure_ascii=False))

In [ ]:
# 7) Render real + progreso en vivo
import os, json, time, subprocess, pathlib, shutil
out=pathlib.Path(PROJECT_DIR)/'out'
out.mkdir(exist_ok=True)
progress=out/'render-progress.json'
if progress.exists(): progress.unlink()

env=os.environ.copy()
env['BLENDER_PROJECT_SLUG']=PROJECT_SLUG
env['BLENDER_PROGRESS_FILE']=str(progress)
# IMPORTANTE: no activamos LIBGL_ALWAYS_SOFTWARE; queremos comprobar el backend real de Colab.

cmd=[BLENDER_BIN,'-b','--python','/content/Chat-gpt-work-/scripts/blender/render-with-progress.py','--',f"{PROJECT_DIR}/{cfg.get('sceneScript','scene.py')}"]
print('Ejecutando:', ' '.join(cmd))
start=time.time()
log_path=out/'colab-render.log'
with open(log_path,'w') as log:
    p=subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, text=True)
    last=None
    while p.poll() is None:
        if progress.exists():
            try:
                d=json.load(open(progress))
                stamp=(d.get('percent'), d.get('current_frame'), d.get('eta_seconds'))
                if stamp != last:
                    print(f"{d.get('percent')}% | frame {d.get('current_frame')}/{d.get('frame_end')} | ETA {d.get('eta_seconds')} s")
                    last=stamp
            except Exception:
                pass
        time.sleep(10)
elapsed=time.time()-start
print('Return code:', p.returncode)
print(f'Tiempo total: {elapsed/60:.2f} min ({elapsed:.1f} s)')
print('Últimas líneas del log:')
print('\n'.join(open(log_path, errors='ignore').read().splitlines()[-40:]))
if p.returncode != 0:
    raise RuntimeError('El render falló; revisa el log mostrado arriba.')

In [ ]:
# 8) Copiar resultado a la misma estructura de Drive
drive_project=cfg.get('driveProjectDir', PROJECT_SLUG)
drive_out=pathlib.Path('/content/drive/MyDrive/Remotion Projects')/drive_project/'renders-colab'
drive_out.mkdir(parents=True, exist_ok=True)
for f in out.iterdir():
    if f.is_file():
        shutil.copy2(f, drive_out/f.name)
print('Copiado a:', drive_out)
print('Archivos:')
for f in sorted(drive_out.iterdir()):
    print(f.name, f.stat().st_size)

## Qué medimos

El dato clave es `Tiempo total` del paso 7. Compáralo con el tiempo del workflow original de GitHub Actions. También revisa el `GPU_RENDERER` del preflight: si Blender no está usando el backend esperado, el resultado sigue siendo útil porque nos dice que debemos ajustar el backend de Colab antes de adoptarlo.